# Slowly Changing Dimensions (SCD)

## Overview
Dimension attributes change over time. A customer's risk tier changes; they move province; their segment is reclassified. SCD types define **how to handle those changes** — discard history (Type 1), preserve it (Type 2), or store one prior value (Type 3).

**SCD decision guide:**
- **Type 1** — History doesn't matter. Overwrite. (Name typos, system corrections)
- **Type 2** — History matters. Preserve full timeline. (Risk tier, pricing band, province for regulatory)
- **Type 3** — Only the prior value matters. One extra column. (A/B migration tracking)

```sql
-- dim_customer SCD2 columns:
customer_key    INTEGER  -- surrogate (new key for each version)
customer_id     TEXT     -- natural key (same across versions)
effective_from  DATE     -- version valid from
effective_to    DATE     -- NULL = current version
is_current      INTEGER  -- 1 = current, 0 = expired
```

---

In [1]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


Data warehouse dataset loaded:
  dim_date              : 1096 rows
  dim_customer          : 6 rows
  dim_product           : 7 rows
  dim_agent             : 4 rows
  fact_policy           : 18 rows

  Total premiums: $45,600


---
## SCD Type 1: overwrite

In [2]:
print("=== SCD Type 1: overwrite (no history) ===")
print()

print("SCD Type 1 — just UPDATE the row:")
print("  Use when: history is not needed; dimension attribute should always reflect current state")
print("  Example: fix a typo in a customer name; update a province after customer moves")
print()

print("Current dim_customer:")
rows = conn.execute("SELECT customer_key, customer_id, full_name, province, risk_tier FROM dim_customer").fetchall()
for r in rows:
    print(f"  {r['customer_key']}  {r['customer_id']}  {r['full_name']:<22s}  {r['province']}  {r['risk_tier']}")

# SCD1 update: Bob moves from BC to AB
conn.execute("""
    UPDATE dim_customer
    SET    province = 'AB'
    WHERE  customer_id = 'C002'
""")
conn.commit()
print()
print("After SCD1 UPDATE (Bob moved from BC to AB):")
rows = conn.execute("SELECT customer_key, customer_id, full_name, province FROM dim_customer WHERE customer_id='C002'").fetchall()
for r in rows:
    print(f"  key={r['customer_key']}  {r['customer_id']}  {r['full_name']}  province={r['province']}")
print("  → History of BC address is LOST (by design for SCD1)")

print()
print("PostgreSQL SCD1 UPSERT pattern:")
print("""
  -- INSERT or UPDATE (upsert) using ON CONFLICT:
  INSERT INTO dim_customer (customer_id, full_name, province, risk_tier)
  VALUES ('C002', 'Bob Harrington', 'AB', 'medium')
  ON CONFLICT (customer_id)
  DO UPDATE SET
      province  = EXCLUDED.province,
      risk_tier = EXCLUDED.risk_tier;
  -- EXCLUDED refers to the proposed insert row
""")


=== SCD Type 1: overwrite (no history) ===

SCD Type 1 — just UPDATE the row:
  Use when: history is not needed; dimension attribute should always reflect current state
  Example: fix a typo in a customer name; update a province after customer moves

Current dim_customer:
  1  C001  Alice Nguyen            ON  low
  2  C002  Bob Harrington          BC  medium
  3  C003  Fatima Al-Rashid        QC  low
  4  C004  James MacLeod           NS  high
  5  C005  Mei-Ling Chen           AB  medium
  6  C006  David Park              ON  low

After SCD1 UPDATE (Bob moved from BC to AB):
  key=2  C002  Bob Harrington  province=AB
  → History of BC address is LOST (by design for SCD1)

PostgreSQL SCD1 UPSERT pattern:

  -- INSERT or UPDATE (upsert) using ON CONFLICT:
  INSERT INTO dim_customer (customer_id, full_name, province, risk_tier)
  VALUES ('C002', 'Bob Harrington', 'AB', 'medium')
  ON CONFLICT (customer_id)
  DO UPDATE SET
      province  = EXCLUDED.province,
      risk_tier = EXCLUDED.r

---
## SCD Type 2: full history

In [3]:
print("=== SCD Type 2: add a new row (full history) ===")
print()

print("SCD Type 2 — expire old row, insert new row with new surrogate key")
print("  Use when: history matters (premium pricing is based on historical province)")
print("  Columns: effective_from, effective_to, is_current")
print()

# Show current state
print("Before SCD2 change:")
rows = conn.execute("""
    SELECT customer_key, customer_id, full_name, risk_tier, effective_from, effective_to, is_current
    FROM dim_customer WHERE customer_id='C004'
""").fetchall()
for r in rows:
    print(f"  key={r['customer_key']}  {r['full_name']}  risk={r['risk_tier']}  "
          f"from={r['effective_from']}  to={r['effective_to']}  current={r['is_current']}")

# James MacLeod's risk tier changes from 'high' to 'medium'
change_date = '2024-06-01'

# Step 1: expire current row
conn.execute("""
    UPDATE dim_customer
    SET    effective_to = ?,
           is_current   = 0
    WHERE  customer_id = 'C004'
    AND    is_current  = 1
""", (change_date,))

# Step 2: insert new row (new surrogate key)
conn.execute("""
    INSERT INTO dim_customer (customer_id, full_name, province, city, segment, risk_tier, effective_from, effective_to, is_current)
    SELECT customer_id, full_name, province, city, segment,
           'medium',        -- new risk tier
           ?,               -- effective_from = change date
           NULL,            -- effective_to = open (current)
           1                -- is_current = 1
    FROM   dim_customer
    WHERE  customer_id = 'C004'
    AND    is_current  = 0
    AND    effective_to = ?
""", (change_date, change_date))
conn.commit()

print()
print(f"After SCD2 change (James risk_tier high→medium, effective {change_date}):")
rows = conn.execute("""
    SELECT customer_key, customer_id, full_name, risk_tier, effective_from, effective_to, is_current
    FROM dim_customer WHERE customer_id='C004'
    ORDER BY customer_key
""").fetchall()
for r in rows:
    status = '← CURRENT' if r['is_current'] else '← EXPIRED'
    print(f"  key={r['customer_key']}  risk={r['risk_tier']:<8s}  "
          f"from={r['effective_from']}  to={str(r['effective_to']):<12s}  {status}")

print()
print("Querying SCD2 — get current version only:")
print("  WHERE is_current = 1")
print("  OR: WHERE effective_to IS NULL")
print()
print("Querying SCD2 — point-in-time (as-of a specific date):")
print("  WHERE effective_from <= '2023-12-31'")
print("    AND (effective_to > '2023-12-31' OR effective_to IS NULL)")
print()
print("PostgreSQL SCD2 MERGE pattern (PostgreSQL 15+):")
print("""
  MERGE INTO dim_customer AS target
  USING (SELECT 'C004' AS customer_id, 'medium' AS risk_tier, '2024-06-01'::date AS eff_from) AS src
  ON target.customer_id = src.customer_id AND target.is_current = true
  WHEN MATCHED AND target.risk_tier <> src.risk_tier THEN
      UPDATE SET is_current = false, effective_to = src.eff_from
  WHEN NOT MATCHED THEN
      INSERT (customer_id, risk_tier, effective_from, is_current)
      VALUES (src.customer_id, src.risk_tier, src.eff_from, true);
  -- Note: MERGE handles expire but not the new-row insert in one statement
  -- Typically done in two steps or a stored procedure
""")


=== SCD Type 2: add a new row (full history) ===

SCD Type 2 — expire old row, insert new row with new surrogate key
  Use when: history matters (premium pricing is based on historical province)
  Columns: effective_from, effective_to, is_current

Before SCD2 change:
  key=4  James MacLeod  risk=high  from=2022-01-01  to=None  current=1

After SCD2 change (James risk_tier high→medium, effective 2024-06-01):
  key=4  risk=high      from=2022-01-01  to=2024-06-01    ← EXPIRED
  key=7  risk=medium    from=2024-06-01  to=None          ← CURRENT

Querying SCD2 — get current version only:
  WHERE is_current = 1
  OR: WHERE effective_to IS NULL

Querying SCD2 — point-in-time (as-of a specific date):
  WHERE effective_from <= '2023-12-31'
    AND (effective_to > '2023-12-31' OR effective_to IS NULL)

PostgreSQL SCD2 MERGE pattern (PostgreSQL 15+):

  MERGE INTO dim_customer AS target
  USING (SELECT 'C004' AS customer_id, 'medium' AS risk_tier, '2024-06-01'::date AS eff_from) AS src
  ON targe

---
## SCD Type 3 and comparison

In [4]:
print("=== SCD Type 3: previous value column ===")
print()

print("SCD Type 3 — add a column for the previous value")
print("  Use when: only ONE prior value matters; history depth is always exactly 1")
print("  Example: track both current and previous risk tier for a customer")
print()

# Add SCD3 columns (simulate with a view since we can't ALTER easily mid-notebook)
print("SCD3 schema (add prev_ columns):")
print("""
  ALTER TABLE dim_customer
      ADD COLUMN prev_risk_tier TEXT,
      ADD COLUMN risk_tier_changed_date TEXT;
""")
print()

# Demonstrate the update
print("SCD3 update logic:")
print("""
  UPDATE dim_customer
  SET    prev_risk_tier         = risk_tier,        -- save current as previous
         risk_tier_changed_date = '2024-06-01',
         risk_tier              = 'medium'           -- overwrite with new
  WHERE  customer_id = 'C004';
""")
print()
print("After SCD3 update:")
print("  customer_id  risk_tier  prev_risk_tier  risk_tier_changed_date")
print("  C004         medium     high            2024-06-01")
print()

comparison = [
    ("SCD1", "Overwrite",             "No history",   "Name corrections, data fixes"),
    ("SCD2", "New row + expire",      "Full history", "Pricing, compliance, audit trail"),
    ("SCD3", "Prev-value column",     "1 prior value","Trend analysis, A/B tracking"),
    ("SCD4", "History table",         "Full history", "Separate audit/history table"),
    ("SCD6", "SCD1+2+3 combined",     "Full + current","Hybrid: current + prior + history row"),
]
print("SCD type comparison:")
print(f"  {'Type':<6s}  {'Mechanism':<22s}  {'History':<14s}  Best for")
print("  " + "-"*68)
for t, mech, hist, use in comparison:
    print(f"  {t:<6s}  {mech:<22s}  {hist:<14s}  {use}")


=== SCD Type 3: previous value column ===

SCD Type 3 — add a column for the previous value
  Use when: only ONE prior value matters; history depth is always exactly 1
  Example: track both current and previous risk tier for a customer

SCD3 schema (add prev_ columns):

  ALTER TABLE dim_customer
      ADD COLUMN prev_risk_tier TEXT,
      ADD COLUMN risk_tier_changed_date TEXT;


SCD3 update logic:

  UPDATE dim_customer
  SET    prev_risk_tier         = risk_tier,        -- save current as previous
         risk_tier_changed_date = '2024-06-01',
         risk_tier              = 'medium'           -- overwrite with new
  WHERE  customer_id = 'C004';


After SCD3 update:
  customer_id  risk_tier  prev_risk_tier  risk_tier_changed_date
  C004         medium     high            2024-06-01

SCD type comparison:
  Type    Mechanism               History         Best for
  --------------------------------------------------------------------
  SCD1    Overwrite               No history     

---
## Common Pitfalls

**1. Not handling SCD2 when querying historical facts**
A customer has two SCD2 rows (key=4, risk='high') and (key=7, risk='medium'). A fact from 2022 points to key=4. Querying `WHERE is_current = 1` returns key=7 and misattributes 2022 facts to 'medium' risk. Always join fact to dimension without filtering on `is_current` unless you specifically want current attributes only.

**2. Forgetting to set effective_to on the expired row**
SCD2 requires the old row's `effective_to` to be set before inserting the new row. If both steps aren't in the same transaction, a failure between steps leaves two current rows for the same customer, causing double-counting.

**3. Using SCD2 for volatile attributes**
If a customer's `last_login_date` changes daily and it's in an SCD2 dimension, you'll have millions of version rows per customer within a year. Volatile attributes belong in separate event/activity tables, not SCD2 dimensions.

**4. Mixing SCD types within one dimension**
Some attributes in dim_customer can be SCD1 (fix typos: full_name) while others are SCD2 (history matters: risk_tier). This is valid but requires careful ETL logic. Clearly document which attributes are SCD1 vs SCD2 in the DWH data dictionary.

**5. SCD2 fact-to-dimension join returning duplicate rows**
Without `AND is_current = 1` in the dimension lookup during ETL, a customer with 3 SCD2 versions joins to 3 dimension rows, tripling the fact row in results. When looking up the current surrogate for new facts, always filter on `is_current = 1`.

**6. Incorrect date range logic for point-in-time queries**
`WHERE effective_from <= '2023-06-01' AND effective_to > '2023-06-01'` fails for the current row if `effective_to IS NULL`. Correct form: `WHERE effective_from <= '2023-06-01' AND (effective_to > '2023-06-01' OR effective_to IS NULL)`.


---
*sql_methods_library - Samantha McGarrigle*